# AIC 2026 — OCR ingestion · HunyuanOCR-1.5

Notebook nguồn chạy độc lập trên **Google Colab** hoặc **Kaggle**. Job đọc đúng
keyframe do `aic2026-keyframes-transnetv2` cung cấp, chạy text spotting bằng
`tencent/HunyuanOCR` (VLM OCR ~1B tham số) và ghi cho mỗi video:

- `ocr.parquet`: đúng một dòng cho mỗi keyframe, gồm text gộp, trạng thái và
  output thô của model để có thể parse lại mà không chạy GPU;
- `ocr_regions.parquet`: một dòng cho mỗi dòng chữ, có bounding box chuẩn hóa;
- `_OCR_SUCCESS.json`: marker provenance/hash, upload cuối cùng trong cùng commit.

Cách chạy theo cấu hình Tencent đã kiểm chứng cho CUDA 12: **vLLM 0.18.1**, mỗi GPU
một server OpenAI-compatible chạy trong tiến trình con. Notebook không import torch
hay transformers, chỉ gửi request HTTP, nên không xung đột thư viện với vLLM.
Không cài transformers 5.x: bản đó cần cho đường transformers-native nhưng làm hỏng
HunYuanVL trong vLLM 0.18.1.

Sampling giữ đúng khuyến nghị chính thức: greedy, `repetition_penalty=1.08`. Output
bị cắt vì hết token hoặc lặp đuôi được **đánh dấu trạng thái**, không bị coi như
frame không có chữ.

Quy trình an toàn:

1. Chạy cell cài đặt; nếu nó yêu cầu restart thì restart runtime rồi chạy lại từ đầu.
2. Chạy toàn bộ với `PREFLIGHT_ONLY=True`; xem ảnh có box, trạng thái và tốc độ đo được.
3. Đặt `PREFLIGHT_ONLY=False`, giữ `MAX_VIDEOS_PER_RUN=5` để chạy pilot.
4. Sau khi kiểm tra artifact, pin revision in ra ở cell cấu hình và đặt
   `MAX_VIDEOS_PER_RUN=None` để chạy toàn bộ.
5. Nếu runtime ngắt, chạy lại từ đầu. Marker hợp lệ được skip; checkpoint cục bộ
   tiếp tục video dang dở nếu storage runtime còn tồn tại.

In [ ]:
# HunyuanOCR-1.5 được Tencent kiểm chứng với vLLM 0.18.1 (CUDA 12). vLLM tự kéo
# torch 2.10 và transformers 4.57; không cài transformers 5.x vào môi trường này.
import importlib.metadata
import subprocess
import sys


def installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None


REQUIRED_VLLM = "0.18.1"
if installed_version("vllm") != REQUIRED_VLLM:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", f"vllm=={REQUIRED_VLLM}",
    ])
    print(
        "✅ ĐÃ CÀI XONG — ĐÂY KHÔNG PHẢI LỖI.\n"
        "vLLM vừa thay torch/transformers nên cần nạp lại Python:\n"
        "  • Colab: Runtime → Restart session (không chọn Disconnect and delete runtime)\n"
        "  • Kaggle: Run → Restart & clear cell outputs\n"
        "Sau đó chạy lại notebook từ cell đầu tiên; lần sau cell này sẽ không cài lại.\n"
        "Notebook dừng tại đây có chủ đích để các cell sau không chạy với thư viện cũ."
    )
    raise SystemExit(0)
TRANSFORMERS_VERSION = installed_version("transformers")
if TRANSFORMERS_VERSION is None or int(TRANSFORMERS_VERSION.split(".")[0]) >= 5:
    raise RuntimeError(
        f"vLLM {REQUIRED_VLLM} cần transformers<5, đang có {TRANSFORMERS_VERSION}"
    )
print({
    name: installed_version(name)
    for name in ("vllm", "torch", "transformers", "huggingface_hub", "pandas", "pyarrow")
})

In [ ]:
import base64
import gc
import hashlib
import io
import json
import os
import re
import shutil
import subprocess
import sys
import tarfile
import tempfile
import threading
import time
import unicodedata
from collections import Counter, defaultdict
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, as_completed, wait
from datetime import datetime, timezone
from pathlib import Path, PurePosixPath

os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "180"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "45"

import pandas as pd
import requests
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download, snapshot_download
from PIL import Image, ImageDraw, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False

INPUT_REPO = "aqpahm/aic2026-keyframes-transnetv2"
OUTPUT_REPO = "aqpahm/aic2026-ocr-hunyuanocr"
MODEL_ID = "tencent/HunyuanOCR"
MODEL_REVISION_PIN = None
INPUT_REVISION_PIN = None
OUTPUT_PRIVATE = True

PREFLIGHT_ONLY = True
RUN_SMOKE_TEST_BEFORE_INGEST = True
PREFLIGHT_FRAMES = 5
PREFLIGHT_FRAME_SPECS = [
    ("L26_V221", 419),
    ("L30_V096", 637),
    ("L30_V096", 646),
]
THROUGHPUT_PROBE_FRAMES = 96
MAX_VIDEOS_PER_RUN = 5
SELECT_LEVELS = None  # Example: {"L21"}; None means every supplied level.

# Official HunyuanOCR-1.5 task prompts (inference/utils/tasks.py). Keep them verbatim:
# the model was trained on these exact instructions.
TASK_PROMPTS = {
    "spotting_hunyuan": "检测并识别图片中的文字，将文本坐标格式化输出。",
    "spotting_json": (
        "检测并识别图中所有的文字行，请按从上到下、从左到右的阅读顺序进行识别。 "
        "输出格式为 JSON 数组，每个元素必须包含："
        '"box": [xmin, ymin, xmax, ymax]（坐标需归一化到 [0, 1000] 范围内）；'
        '"text": "识别出的文字内容"。 '
        "注意：请直接输出 JSON 数组，不要包含任何多余的描述性文字。"
    ),
}
TASK_TYPE = "spotting_hunyuan"
PROMPT = TASK_PROMPTS[TASK_TYPE]
COORD_SCALE = 1000  # Spotting boxes are normalized to [0, 1000].

# Official sampling: greedy + repetition_penalty=1.08. Keyframe spotting output is
# short, so a small token cap bounds the cost of a degenerate repetition loop.
TEMPERATURE = 0.0
TOP_P = 1.0
TOP_K = -1
REPETITION_PENALTY = 1.08
MAX_NEW_TOKENS = 3072
TAIL_REPEAT_MIN_REPEATS = 8

# vLLM server per GPU. T4 has no bfloat16, so float16 is the default for every
# runtime to keep outputs comparable. If preflight shows garbage or empty output on
# frames that clearly contain text, switch to "float32" (slower, but numerically safe).
VLLM_DTYPE = "float16"
VLLM_BASE_PORT = 8000
GPU_MEMORY_UTILIZATION = 0.90
# The model's processor allows up to 4096x4096 pixels (~16.9k image tokens in vLLM's
# worst case), which cannot fit a T4-sized context. 1920x1080 keeps 720p/1080p
# keyframes at native resolution and caps one image at ~2.2k tokens.
MAX_PIXELS = 1920 * 1080
MAX_MODEL_LEN = 8192
MAX_NUM_SEQS = 48
REQUESTS_IN_FLIGHT_PER_GPU = 48
REQUEST_TIMEOUT_SEC = 600
REQUEST_ATTEMPTS = 4
SERVER_START_TIMEOUT_SEC = 1200

UPLOAD_WORKERS = 1
MAX_PENDING_UPLOADS = 2
CHECKPOINT_FSYNC_EVERY_FRAMES = 25
MAX_CONSECUTIVE_IDENTICAL_FAILURES = 3
SCHEMA_VERSION = 1
NO_TEXT_SENTINEL = "<NO_TEXT>"  # Display only; stored OCR text is empty.
OCR_STATUSES = {"ok", "no_text", "truncated", "repetition", "parse_error"}


def runtime_name():
    if "google.colab" in sys.modules:
        return "colab"
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or Path("/kaggle").exists():
        return "kaggle"
    return "local"


def secret(name):
    if os.environ.get(name):
        return os.environ[name]
    try:
        if RUNTIME == "colab":
            from google.colab import userdata
            return userdata.get(name)
        if RUNTIME == "kaggle":
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
    except Exception as error:
        raise RuntimeError(f"Cannot read {name}; grant this notebook access") from error
    raise RuntimeError(f"Missing {name}; add it to Colab/Kaggle Secrets")


KNOWN_COMPUTE_CAPABILITY = {"P100": 6.0, "V100": 7.0, "T4": 7.5, "A100": 8.0, "A10": 8.6, "L4": 8.9, "H100": 9.0}


def query_gpus():
    """List GPUs through nvidia-smi so the notebook never initializes CUDA itself."""

    def run_query(fields):
        return subprocess.run(
            ["nvidia-smi", f"--query-gpu={fields}", "--format=csv,noheader,nounits"],
            check=True, capture_output=True, text=True,
        ).stdout.strip().splitlines()

    try:
        try:
            rows = [line.split(",") for line in run_query("index,name,memory.total,compute_cap")]
        except subprocess.CalledProcessError:
            # Older drivers do not know the compute_cap field; infer it from the GPU name.
            rows = [line.split(",") for line in run_query("index,name,memory.total")]
    except FileNotFoundError as error:
        raise RuntimeError("Enable a GPU runtime before running OCR") from error
    gpus = []
    for row in rows:
        index, name, memory = (part.strip() for part in row[:3])
        if len(row) > 3:
            capability = float(row[3])
        else:
            matches = [cc for key, cc in KNOWN_COMPUTE_CAPABILITY.items() if key in name]
            if not matches:
                raise RuntimeError(f"Cannot determine compute capability of {name}")
            capability = matches[0]
        gpus.append({
            "index": int(index),
            "name": name,
            "compute_capability": capability,
            "memory_mib": int(float(memory)),
        })
    return gpus


RUNTIME = runtime_name()
ROOT = (
    Path("/content/aic-ocr-hunyuanocr") if RUNTIME == "colab"
    else Path("/kaggle/temp/aic-ocr-hunyuanocr") if RUNTIME == "kaggle"
    else Path(tempfile.gettempdir()) / "aic-ocr-hunyuanocr"
)
DOWNLOAD_ROOT = ROOT / "downloads"
OUTPUT_ROOT = ROOT / "output"
CHECKPOINT_ROOT = ROOT / "checkpoints"
LOG_ROOT = ROOT / "logs"
MODEL_DIR = ROOT / "model"
for directory in (DOWNLOAD_ROOT, OUTPUT_ROOT, CHECKPOINT_ROOT, LOG_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

GPUS = query_gpus()
if not GPUS:
    raise RuntimeError("Enable a GPU runtime before running OCR")
for gpu in GPUS:
    print(f"gpu:{gpu['index']}: {gpu['name']} | sm {gpu['compute_capability']} | {gpu['memory_mib']} MiB")
    if gpu["compute_capability"] < 7.0:
        raise RuntimeError(f"vLLM needs compute capability >= 7.0; {gpu['name']} is unsupported")
    if VLLM_DTYPE == "bfloat16" and gpu["compute_capability"] < 8.0:
        raise RuntimeError(f"{gpu['name']} has no bfloat16; use VLLM_DTYPE='float16' or 'float32'")
GPU_IDS = [gpu["index"] for gpu in GPUS]
if PREFLIGHT_ONLY:
    GPU_IDS = GPU_IDS[:1]
    print("Preflight uses gpu:0 only; pilot/full starts one vLLM server on every GPU.")

HF_TOKEN = secret("HF_TOKEN")
api = HfApi(token=HF_TOKEN)
print("Runtime:", RUNTIME, "| HF account:", api.whoami()["name"])
if not PREFLIGHT_ONLY:
    api.create_repo(
        repo_id=OUTPUT_REPO,
        repo_type="dataset",
        private=OUTPUT_PRIVATE,
        exist_ok=True,
    )
INPUT_REVISION = INPUT_REVISION_PIN or api.dataset_info(INPUT_REPO).sha
MODEL_REVISION = MODEL_REVISION_PIN or api.model_info(MODEL_ID).sha
# The repo also ships the archived v1.0 weights and a DFlash draft; the AR server
# only needs the 1.5 base model.
snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    token=HF_TOKEN,
    local_dir=MODEL_DIR,
    ignore_patterns=["v1.0/*", "dflash/*", "assets/*"],
)
for required_file in ("config.json", "model.safetensors", "preprocessor_config.json"):
    if not (MODEL_DIR / required_file).is_file():
        raise RuntimeError(f"Model download is missing {required_file}")

VLLM_VERSION = importlib.metadata.version("vllm")
TORCH_VERSION = importlib.metadata.version("torch")
print("Input revision:", INPUT_REVISION)
print("Model revision:", MODEL_REVISION)
print("vLLM/torch/transformers:", VLLM_VERSION, TORCH_VERSION, TRANSFORMERS_VERSION)
print("Output:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")

RUN_CONFIG = {
    "source_revision": INPUT_REVISION,
    "model_id": MODEL_ID,
    "model_revision": MODEL_REVISION,
    "vllm_version": VLLM_VERSION,
    "torch_version": TORCH_VERSION,
    "transformers_version": TRANSFORMERS_VERSION,
    "dtype": VLLM_DTYPE,
    "task_type": TASK_TYPE,
    "prompt": PROMPT,
    "coord_scale": COORD_SCALE,
    "temperature": TEMPERATURE,
    "top_p": TOP_P,
    "top_k": TOP_K,
    "repetition_penalty": REPETITION_PENALTY,
    "max_new_tokens": MAX_NEW_TOKENS,
    "max_model_len": MAX_MODEL_LEN,
    "max_pixels": MAX_PIXELS,
    "tail_repeat_min_repeats": TAIL_REPEAT_MIN_REPEATS,
    "schema_version": SCHEMA_VERSION,
}
RUN_CONFIG_SHA256 = hashlib.sha256(
    json.dumps(RUN_CONFIG, sort_keys=True, ensure_ascii=False).encode("utf-8")
).hexdigest()

## 1. Khởi động một vLLM server trên mỗi GPU

Mỗi server là một tiến trình con gắn với đúng một GPU qua `CUDA_VISIBLE_DEVICES`,
giống `inference/vLLM/serve.sh` của Tencent. Nếu server chết khi khởi động, cell in
phần cuối log để thấy ngay nguyên nhân (thiếu VRAM, dtype, lỗi CUDA...) thay vì treo.
Chạy lại cell này sẽ dừng server cũ trước khi mở server mới.

In [ ]:
SERVED_MODEL_NAME = "hunyuanocr"


def log_tail(path, lines=25, max_error_lines=40):
    """Show the root cause first: vLLM's API server only re-raises 'Engine core
    initialization failed', while the real error is logged earlier by EngineCore."""
    try:
        all_lines = Path(path).read_text(encoding="utf-8", errors="replace").splitlines()
    except OSError:
        return "<no log>"
    error_pattern = re.compile(r"Error|Exception|error:|CUDA|out of memory|not supported", re.IGNORECASE)
    root_cause = [
        line for line in all_lines
        if error_pattern.search(line) and not line.lstrip().startswith("(APIServer")
    ][:max_error_lines]
    sections = []
    if root_cause:
        sections.append("--- likely root cause (non-APIServer error lines) ---\n" + "\n".join(root_cause))
    sections.append(f"--- last {lines} log lines ---\n" + "\n".join(all_lines[-lines:]))
    sections.append(f"Full log: {path}")
    return "\n".join(sections)


def stop_servers(servers):
    for server in servers:
        process = server["process"]
        if process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.kill()
        server["log"].close()
    servers.clear()


def vllm_executable():
    path = shutil.which("vllm") or str(Path(sys.executable).with_name("vllm"))
    if not Path(path).exists():
        raise RuntimeError("Cannot find the vllm CLI; rerun the install cell and restart")
    return path


def start_server(gpu_id, port):
    log_path = LOG_ROOT / f"vllm-gpu{gpu_id}-port{port}.log"
    log = log_path.open("w", encoding="utf-8")
    env = dict(
        os.environ,
        CUDA_VISIBLE_DEVICES=str(gpu_id),
        HF_HUB_OFFLINE="1",
        TRANSFORMERS_OFFLINE="1",
    )
    command = [
        vllm_executable(), "serve", str(MODEL_DIR),
        "--served-model-name", SERVED_MODEL_NAME,
        "--host", "127.0.0.1",
        "--port", str(port),
        "--trust-remote-code",
        "--dtype", VLLM_DTYPE,
        "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--max-num-batched-tokens", str(MAX_MODEL_LEN),
        "--max-num-seqs", str(MAX_NUM_SEQS),
        "--limit-mm-per-prompt", json.dumps({"image": 1, "video": 0}),
        "--mm-processor-kwargs", json.dumps({"max_pixels": MAX_PIXELS}),
        "--seed", "0",
    ]
    process = subprocess.Popen(
        command, stdout=log, stderr=subprocess.STDOUT, env=env, start_new_session=True,
    )
    return {"gpu_id": gpu_id, "port": port, "process": process, "log": log, "log_path": log_path}


def wait_until_ready(server):
    url = f"http://127.0.0.1:{server['port']}/v1/models"
    deadline = time.monotonic() + SERVER_START_TIMEOUT_SEC
    while time.monotonic() < deadline:
        if server["process"].poll() is not None:
            raise RuntimeError(
                f"vLLM on gpu:{server['gpu_id']} exited with code "
                f"{server['process'].returncode}. Log tail:\n{log_tail(server['log_path'])}"
            )
        try:
            response = requests.get(url, timeout=5)
            served = {item.get("id") for item in response.json().get("data", [])}
            if response.ok and SERVED_MODEL_NAME in served:
                return
        except (requests.RequestException, ValueError):
            pass
        time.sleep(5)
    raise TimeoutError(
        f"vLLM on gpu:{server['gpu_id']} not ready after {SERVER_START_TIMEOUT_SEC}s. "
        f"Log tail:\n{log_tail(server['log_path'])}"
    )


stop_servers(globals().get("SERVERS", []))
# Also clear servers orphaned by an interrupted earlier run (same commands as Tencent's docs).
subprocess.run(["pkill", "-9", "-f", "vllm serve"], check=False)
subprocess.run(["pkill", "-9", "-f", "VLLM::EngineCore"], check=False)
time.sleep(3)
SERVERS = [start_server(gpu_id, VLLM_BASE_PORT + gpu_id) for gpu_id in GPU_IDS]
started = time.perf_counter()
for server in SERVERS:
    print(f"Waiting for vLLM on gpu:{server['gpu_id']} (port {server['port']}) ...")
    wait_until_ready(server)
print(f"{len(SERVERS)} vLLM server(s) ready in {time.perf_counter() - started:.0f}s")

REQUEST_POOL = ThreadPoolExecutor(max_workers=REQUESTS_IN_FLIGHT_PER_GPU * len(SERVERS))

## 2. Gửi request và parse output spotting

Parser hỗ trợ bốn định dạng spotting mà Tencent dùng khi chấm điểm
(`train_verl/reward/task_scorers/spotting.py`): `text(x1,y1),(x2,y2)` của chế độ
`spotting_hunyuan`, mảng JSON của `spotting_json`, và hai dạng `<ref>/<quad>`.
Tọa độ nằm trong `[0, 1000]` và được đổi về `[0, 1]`.

Trạng thái mỗi frame:

- `ok`: có ít nhất một vùng chữ hợp lệ;
- `no_text`: model trả rỗng hoặc mảng rỗng;
- `truncated`: hết `MAX_NEW_TOKENS` trước khi xong, vùng chữ đã parse vẫn được giữ;
- `repetition`: đuôi output lặp một đơn vị ngắn (dấu hiệu thoái hóa);
- `parse_error`: có output nhưng không đọc được vùng chữ hợp lệ nào.

Lỗi HTTP hoặc server chết làm video thất bại, không bị biến thành `no_text`.

In [ ]:
def has_tail_repetition(text, min_repeats=8, max_unit=256):
    """Tencent's tail-repetition check: the tail is one short unit repeated."""
    length = len(text)
    if length < min_repeats * 2:
        return False
    upper = min(max_unit, length // min_repeats)
    for unit_length in range(1, upper + 1):
        unit = text[-unit_length:]
        if not unit.strip():
            continue
        if all(
            text[-unit_length * k:-unit_length * (k - 1)] == unit
            for k in range(2, min_repeats + 1)
        ):
            return True
    return False


def parse_spotting_output(text):
    """Return (format, [(text, x1, y1, x2, y2), ...]) with coordinates on the model scale."""
    stripped = text.strip()
    if not stripped:
        return "empty", []
    fenced = re.fullmatch(r"```(?:json)?\s*(.*?)\s*```", stripped, flags=re.DOTALL)
    if fenced:
        stripped = fenced.group(1).strip()
    if stripped.startswith("["):
        try:
            items = json.loads(stripped)
        except json.JSONDecodeError:
            items = None
        if isinstance(items, list):
            regions = []
            for item in items:
                if not isinstance(item, dict):
                    continue
                box = item.get("box", item.get("bbox"))
                if not isinstance(box, list | tuple) or len(box) != 4:
                    continue
                try:
                    x1, y1, x2, y2 = (float(value) for value in box)
                except (TypeError, ValueError):
                    continue
                regions.append((str(item.get("text", "")), x1, y1, x2, y2))
            return "json", regions
    if "<ref>" in stripped and "<quad>(" in stripped:
        pattern = re.compile(r"<ref>(.*?)</ref>\s*<quad>\((\d+),(\d+)\),\((\d+),(\d+)\)</quad>", re.DOTALL)
        return "ref_quad", [
            (match[0], *(float(value) for value in match[1:])) for match in pattern.findall(stripped)
        ]
    if "<quad><pos_" in stripped:
        pattern = re.compile(r"<quad><pos_(\d+)><pos_(\d+)><pos_(\d+)><pos_(\d+)></quad><ref>(.*?)</ref>", re.DOTALL)
        return "quad_ref", [
            (match[4], *(float(value) for value in match[:4])) for match in pattern.findall(stripped)
        ]
    # spotting_hunyuan: items "text(x1,y1),(x2,y2)" back to back. Anchor on each
    # coordinate pair so parentheses inside the text ("Hà Nội (TTXVN)") survive.
    coordinates = re.compile(r"\((\d+),(\d+)\),\((\d+),(\d+)\)")
    regions, last = [], 0
    for match in coordinates.finditer(stripped):
        content = stripped[last:match.start()].strip()
        last = match.end()
        if not re.search(r"[^\s(),]", content):
            continue
        regions.append((content, *(float(value) for value in match.groups())))
    if regions:
        return "hunyuan", regions
    return "unknown", []


def normalize_regions(raw_regions, coord_scale):
    """Normalize boxes to [0, 1], keep model reading order, drop empty/invalid/duplicate lines."""
    regions, dropped, seen = [], 0, set()
    for text, x1, y1, x2, y2 in raw_regions:
        text = unicodedata.normalize("NFC", str(text)).strip()
        left, right = sorted(min(max(value / coord_scale, 0.0), 1.0) for value in (x1, x2))
        top, bottom = sorted(min(max(value / coord_scale, 0.0), 1.0) for value in (y1, y2))
        key = (text, round(left, 4), round(top, 4), round(right, 4), round(bottom, 4))
        if not text or right <= left or bottom <= top or key in seen:
            dropped += 1
            continue
        seen.add(key)
        regions.append({
            "text": text,
            "bbox_x1": left,
            "bbox_y1": top,
            "bbox_x2": right,
            "bbox_y2": bottom,
        })
    for index, region in enumerate(regions):
        region["region_id"] = index
        region["reading_order"] = index
    return regions, dropped


def build_frame_result(response, width, height, coord_scale, min_repeats):
    raw_output = response["raw_output"]
    spotting_format, raw_regions = parse_spotting_output(raw_output)
    regions, dropped = normalize_regions(raw_regions, coord_scale)
    if response["finish_reason"] == "length":
        status = "truncated"
    elif has_tail_repetition(raw_output, min_repeats=min_repeats):
        status = "repetition"
    elif regions:
        status = "ok"
    elif spotting_format in {"empty", "json"} and not raw_regions:
        status = "no_text"
    else:
        status = "parse_error"
    return {
        "ocr_text": "\n".join(region["text"] for region in regions),
        "ocr_status": status,
        "regions": regions,
        "dropped_region_count": dropped,
        "spotting_format": spotting_format,
        "raw_output": raw_output,
        "finish_reason": response["finish_reason"],
        "completion_tokens": response["completion_tokens"],
        "latency_sec": response["latency_sec"],
        "image_width": width,
        "image_height": height,
    }


class FatalRequestError(RuntimeError):
    pass


_thread_state = threading.local()


def http_session():
    session = getattr(_thread_state, "session", None)
    if session is None:
        session = requests.Session()
        _thread_state.session = session
    return session


def image_data_url(image_bytes):
    with Image.open(io.BytesIO(image_bytes)) as source:
        source.load()
        width, height = source.size
        image_format = (source.format or "").upper()
        if image_format in {"JPEG", "PNG"}:
            data, mime = image_bytes, f"image/{image_format.lower()}"
        else:
            buffer = io.BytesIO()
            source.convert("RGB").save(buffer, format="PNG")
            data, mime = buffer.getvalue(), "image/png"
    return f"data:{mime};base64,{base64.b64encode(data).decode('ascii')}", width, height


def request_ocr(data_url, port):
    payload = {
        "model": SERVED_MODEL_NAME,
        "messages": [
            {"role": "system", "content": ""},
            {"role": "user", "content": [
                {"type": "image_url", "image_url": {"url": data_url}},
                {"type": "text", "text": PROMPT},
            ]},
        ],
        "max_tokens": MAX_NEW_TOKENS,
        "temperature": TEMPERATURE,
        "top_p": TOP_P,
        "top_k": TOP_K,
        "repetition_penalty": REPETITION_PENALTY,
        "skip_special_tokens": True,
        "seed": 0,
    }
    started = time.perf_counter()
    response = http_session().post(
        f"http://127.0.0.1:{port}/v1/chat/completions", json=payload, timeout=REQUEST_TIMEOUT_SEC,
    )
    if 400 <= response.status_code < 500:
        raise FatalRequestError(f"vLLM HTTP {response.status_code}: {response.text[:1000]}")
    response.raise_for_status()
    body = response.json()
    choice = body["choices"][0]
    return {
        "raw_output": choice["message"].get("content") or "",
        "finish_reason": choice.get("finish_reason"),
        "completion_tokens": int((body.get("usage") or {}).get("completion_tokens") or 0),
        "latency_sec": round(time.perf_counter() - started, 3),
    }


def ocr_frame(image_bytes, server):
    data_url, width, height = image_data_url(image_bytes)
    for attempt in range(1, REQUEST_ATTEMPTS + 1):
        if server["process"].poll() is not None:
            raise FatalRequestError(
                f"vLLM on gpu:{server['gpu_id']} died. Log tail:\n{log_tail(server['log_path'])}"
            )
        try:
            response = request_ocr(data_url, server["port"])
            break
        except FatalRequestError:
            raise
        except (requests.RequestException, ValueError, KeyError) as error:
            if attempt == REQUEST_ATTEMPTS:
                raise
            delay = 5 * 2 ** (attempt - 1)
            print(f"  request retry in {delay}s after {type(error).__name__}: {error}")
            time.sleep(delay)
    return build_frame_result(response, width, height, COORD_SCALE, TAIL_REPEAT_MIN_REPEATS)

## 3. Đọc keyframe và checkpoint

Checkpoint có fingerprint của input, model, tham số và schema. Request chạy song song
nhưng checkpoint chỉ ghi phần tiền tố liên tục theo thứ tự frame, nên resume luôn
đúng thứ tự. Lỗi request làm video fail và không có marker thành công.

In [ ]:
def retry(operation, description, attempts=6):
    for attempt in range(1, attempts + 1):
        try:
            return operation()
        except Exception as error:
            if attempt == attempts:
                raise
            delay = min(180, 5 * 2 ** (attempt - 1))
            print(f"{description}: {type(error).__name__}; retry in {delay}s [{attempt}/{attempts}]")
            time.sleep(delay)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        while block := stream.read(8 * 1024 * 1024):
            digest.update(block)
    return digest.hexdigest()


def download_input(filename, local_dir):
    local_dir.mkdir(parents=True, exist_ok=True)
    return Path(retry(
        lambda: hf_hub_download(
            repo_id=INPUT_REPO,
            repo_type="dataset",
            filename=filename,
            revision=INPUT_REVISION,
            token=HF_TOKEN,
            local_dir=local_dir,
        ),
        f"download {filename}",
    ))


def normalized_tar_name(value):
    return PurePosixPath(str(value).replace("\\", "/")).as_posix().lstrip("./")


def tar_lookup(archive):
    by_path = {}
    by_name = defaultdict(list)
    for member in archive.getmembers():
        if member.isfile():
            name = normalized_tar_name(member.name)
            by_path[name] = member
            by_name[PurePosixPath(name).name].append(member)
    return by_path, by_name


def read_frame_bytes(archive, image_path, by_path, by_name):
    expected = normalized_tar_name(image_path)
    member = by_path.get(expected)
    if member is None:
        matches = by_name.get(PurePosixPath(expected).name, [])
        if len(matches) != 1:
            raise RuntimeError(f"Cannot uniquely find {image_path!r} in keyframes.tar")
        member = matches[0]
    stream = archive.extractfile(member)
    if stream is None:
        raise RuntimeError(f"Cannot read {member.name}")
    return stream.read()


ID_COLUMNS = [
    "video_id", "frame_uid", "sample_n", "frame_idx", "shot_id",
    "timestamp_sec", "image_path",
]
OCR_COLUMNS = ID_COLUMNS + [
    "ocr_text", "ocr_text_normalized", "ocr_text_no_accent", "has_text",
    "ocr_status", "region_count", "dropped_region_count", "image_width", "image_height",
    "spotting_format", "finish_reason", "completion_tokens", "raw_output",
    "model_id", "model_revision",
]
REGION_COLUMNS = ID_COLUMNS + [
    "region_id", "reading_order", "text", "text_normalized", "text_no_accent",
    "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2", "model_id", "model_revision",
]


def make_manifest(source_frame, video_id):
    required = {"video_id", "sample_n", "frame_idx", "shot_id", "timestamp_sec", "image_path"}
    missing = required - set(source_frame.columns)
    if missing or source_frame.empty:
        raise RuntimeError(f"{video_id}: empty/incomplete frames.parquet; missing {sorted(missing)}")
    frame = source_frame.sort_values("sample_n", kind="stable").reset_index(drop=True).copy()
    if frame["video_id"].astype(str).nunique() != 1 or str(frame.iloc[0]["video_id"]) != video_id:
        raise RuntimeError(f"{video_id}: source video_id mismatch")
    if frame["sample_n"].duplicated().any() or frame["frame_idx"].duplicated().any():
        raise RuntimeError(f"{video_id}: duplicate sample_n/frame_idx")
    frame["video_id"] = frame["video_id"].astype(str)
    frame["sample_n"] = frame["sample_n"].astype("int64")
    frame["frame_idx"] = frame["frame_idx"].astype("int64")
    frame["shot_id"] = frame["shot_id"].astype("int64")
    frame["timestamp_sec"] = frame["timestamp_sec"].astype("float64")
    frame["image_path"] = frame["image_path"].astype(str)
    frame["frame_uid"] = [f"{video_id}:{index}" for index in frame["frame_idx"]]
    return frame[ID_COLUMNS]


def normalize_search(text):
    return " ".join(unicodedata.normalize("NFC", text).casefold().split())


def remove_accents(text):
    value = unicodedata.normalize("NFD", text)
    value = "".join(char for char in value if unicodedata.category(char) != "Mn")
    return value.replace("đ", "d").replace("Đ", "D")


def checkpoint_path(video_id):
    return CHECKPOINT_ROOT / f"{video_id}.jsonl"


def checkpoint_header(video_id, manifest):
    return {
        "record_type": "header",
        "video_id": video_id,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "frame_count": len(manifest),
        "first_frame_uid": manifest.iloc[0]["frame_uid"],
        "last_frame_uid": manifest.iloc[-1]["frame_uid"],
    }


def rewrite_checkpoint(path, header, records):
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as stream:
        stream.write(json.dumps(header, ensure_ascii=False) + "\n")
        for record in records:
            stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


CHECKPOINT_RESULT_KEYS = {
    "ocr_text", "ocr_status", "regions", "image_width", "image_height", "raw_output",
}


def load_checkpoint(video_id, manifest):
    path = checkpoint_path(video_id)
    expected_header = checkpoint_header(video_id, manifest)
    if not path.exists():
        rewrite_checkpoint(path, expected_header, [])
        return []
    valid = []
    try:
        with path.open(encoding="utf-8") as stream:
            if json.loads(stream.readline()) != expected_header:
                raise ValueError("checkpoint header does not match this run")
            for position, line in enumerate(stream):
                if not line.strip():
                    continue
                try:
                    record = json.loads(line)
                except json.JSONDecodeError:
                    print(f"  Truncating incomplete checkpoint row {position} for {video_id}")
                    break
                if position >= len(manifest):
                    break
                if record.get("frame_uid") != manifest.iloc[position]["frame_uid"]:
                    raise ValueError(f"checkpoint order mismatch at row {position}")
                result = record.get("result")
                if not isinstance(result, dict) or not CHECKPOINT_RESULT_KEYS <= set(result):
                    raise ValueError(f"invalid checkpoint result at row {position}")
                valid.append(record)
    except (OSError, UnicodeError, json.JSONDecodeError, ValueError) as error:
        print(f"  Ignoring incompatible checkpoint for {video_id}: {error}")
        valid = []
    rewrite_checkpoint(path, expected_header, valid)
    if valid:
        print(f"  Resuming {video_id} at frame {len(valid)}/{len(manifest)}")
    return [record["result"] for record in valid]


def infer_video(archive, manifest, video_id):
    """OCR every frame with bounded concurrency; checkpoint the in-order completed prefix."""
    by_path, by_name = tar_lookup(archive)
    results = load_checkpoint(video_id, manifest)
    resumed_frames = len(results)
    total = len(manifest)
    in_flight_limit = REQUESTS_IN_FLIGHT_PER_GPU * len(SERVERS)
    pending, completed = {}, {}
    next_submit = next_write = last_sync = resumed_frames
    started = time.perf_counter()
    stream = checkpoint_path(video_id).open("a", encoding="utf-8")
    try:
        while next_write < total:
            while next_submit < total and len(pending) < in_flight_limit:
                row = manifest.iloc[next_submit]
                image_bytes = read_frame_bytes(archive, row["image_path"], by_path, by_name)
                server = SERVERS[next_submit % len(SERVERS)]
                pending[REQUEST_POOL.submit(ocr_frame, image_bytes, server)] = next_submit
                next_submit += 1
            done, _ = wait(list(pending), return_when=FIRST_COMPLETED)
            for future in done:
                completed[pending.pop(future)] = future.result()
            while next_write in completed:
                result = completed.pop(next_write)
                results.append(result)
                stream.write(json.dumps({
                    "frame_uid": manifest.iloc[next_write]["frame_uid"],
                    "result": result,
                }, ensure_ascii=False) + "\n")
                next_write += 1
                if next_write == total or next_write - last_sync >= CHECKPOINT_FSYNC_EVERY_FRAMES:
                    stream.flush()
                    os.fsync(stream.fileno())
                    last_sync = next_write
                if next_write % 100 == 0 or next_write == total:
                    print(f"  OCR {next_write}/{total}")
    except BaseException:
        for future in pending:
            future.cancel()
        raise
    finally:
        stream.flush()
        os.fsync(stream.fileno())
        stream.close()
    return results, time.perf_counter() - started, resumed_frames


def draw_regions(image, regions):
    preview = image.copy()
    draw = ImageDraw.Draw(preview)
    width, height = preview.size
    line_width = max(2, round(max(width, height) / 500))
    for region in regions:
        draw.rectangle(
            [
                region["bbox_x1"] * width, region["bbox_y1"] * height,
                region["bbox_x2"] * width, region["bbox_y2"] * height,
            ],
            outline="red",
            width=line_width,
        )
    return preview


def preview_jpeg(image_bytes, regions):
    with Image.open(io.BytesIO(image_bytes)) as source:
        image = source.convert("RGB")
    preview = draw_regions(image, regions)
    image.close()
    preview.thumbnail((900, 900), Image.Resampling.LANCZOS)
    buffer = io.BytesIO()
    preview.save(buffer, format="JPEG", quality=85)
    preview.close()
    return buffer.getvalue()


def spread_indices(indices, count):
    take = min(len(indices), max(0, count))
    if take <= 1:
        return indices[:take]
    return [indices[round(i * (len(indices) - 1) / (take - 1))] for i in range(take)]

## 4. Chọn đúng video nguồn và kiểm tra resume

Danh sách video được lấy từ thành viên thực tế của dataset keyframe rồi sắp xếp
theo `level/video_id`; không suy đoán ID liên tục. Chỉ marker có đúng provenance,
cấu hình, schema và hash của cả hai Parquet mới được skip.

In [ ]:
input_files = set(retry(
    lambda: api.list_repo_files(INPUT_REPO, repo_type="dataset", revision=INPUT_REVISION),
    "list keyframe files",
))
output_files = set() if PREFLIGHT_ONLY else set(retry(
    lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
    "list OCR files",
))
pattern = re.compile(r"^data/(L(?:2[1-9]|30))/(L\d+_V\d+)/keyframes\.tar$")
video_specs = []
for filename in sorted(input_files):
    match = pattern.fullmatch(filename)
    if not match:
        continue
    level, video_id = match.groups()
    if SELECT_LEVELS is not None and level not in SELECT_LEVELS:
        continue
    prefix = f"data/{level}/{video_id}"
    frames_filename = f"{prefix}/frames.parquet"
    source_marker = f"{prefix}/_SUCCESS.json"
    if frames_filename not in input_files or source_marker not in input_files:
        raise RuntimeError(f"Incomplete keyframe source for {video_id}")
    video_specs.append({
        "level": level,
        "video_id": video_id,
        "tar_filename": filename,
        "frames_filename": frames_filename,
        "source_marker": source_marker,
    })
if not video_specs:
    raise RuntimeError("No complete keyframe videos found")
spec_by_id = {spec["video_id"]: spec for spec in video_specs}


def remote_complete(spec, files):
    prefix = f"data/{spec['level']}/{spec['video_id']}"
    parquet_name = f"{prefix}/ocr.parquet"
    regions_name = f"{prefix}/ocr_regions.parquet"
    marker_name = f"{prefix}/_OCR_SUCCESS.json"
    if not {parquet_name, regions_name, marker_name} <= files:
        return False
    verify_dir = DOWNLOAD_ROOT / "marker-validation" / spec["video_id"]
    try:
        marker_path = retry(
            lambda: hf_hub_download(
                repo_id=OUTPUT_REPO,
                repo_type="dataset",
                filename=marker_name,
                token=HF_TOKEN,
            ),
            f"download marker {spec['video_id']}",
        )
        marker = json.loads(Path(marker_path).read_text(encoding="utf-8"))
        required = (
            marker.get("status") == "complete"
            and marker.get("video_id") == spec["video_id"]
            and marker.get("source_repo") == INPUT_REPO
            and marker.get("source_revision") == INPUT_REVISION
            and marker.get("model_id") == MODEL_ID
            and marker.get("model_revision") == MODEL_REVISION
            and marker.get("run_config_sha256") == RUN_CONFIG_SHA256
            and int(marker.get("schema_version", -1)) == SCHEMA_VERSION
        )
        if not required:
            return False
        parquet_path = Path(hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            filename=parquet_name,
            token=HF_TOKEN,
            local_dir=verify_dir,
        ))
        regions_path = Path(hf_hub_download(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            filename=regions_name,
            token=HF_TOKEN,
            local_dir=verify_dir,
        ))
        return (
            sha256_file(parquet_path) == marker.get("parquet_sha256")
            and sha256_file(regions_path) == marker.get("regions_parquet_sha256")
        )
    except Exception as error:
        print(f"Invalid remote OCR output for {spec['video_id']}: {error}")
        return False
    finally:
        shutil.rmtree(verify_dir, ignore_errors=True)


if PREFLIGHT_ONLY:
    complete_flags = [False] * len(video_specs)
else:
    with ThreadPoolExecutor(max_workers=12) as validator:
        complete_flags = list(validator.map(
            lambda spec: remote_complete(spec, output_files), video_specs
        ))
missing_specs = [spec for spec, complete in zip(video_specs, complete_flags) if not complete]
selected_specs = (
    missing_specs if MAX_VIDEOS_PER_RUN is None
    else missing_specs[:MAX_VIDEOS_PER_RUN]
)
pending_specs = [] if PREFLIGHT_ONLY else selected_specs
print("Source videos:", len(video_specs))
print("Remote complete:", sum(complete_flags))
print("Selected this run:", len(pending_specs))
if pending_specs:
    print("First/last selected:", pending_specs[0]["video_id"], pending_specs[-1]["video_id"])

## 5. Preflight, ghi artifact và upload

Preflight OCR vài frame cố định để xem box và text, rồi đo tốc độ trên
`THROUGHPUT_PROBE_FRAMES` frame gửi song song. Dataset hiện có khoảng 335 nghìn
keyframe, nên hãy nhìn con số giờ ước tính trước khi chạy full.

Khi ingest, video được xử lý lần lượt nhưng các frame trong video được gửi song
song tới mọi server. Input video kế tiếp được tải trước, còn upload chạy trên thread
riêng. Một commit chứa đủ hai Parquet và marker của đúng một video; marker nằm cuối.

In [ ]:
def build_artifact_frames(manifest, results):
    if len(manifest) != len(results):
        raise RuntimeError("OCR result count does not match keyframe count")
    ocr_rows, region_rows = [], []
    for identity, result in zip(manifest.to_dict("records"), results):
        regions = result["regions"]
        text = result["ocr_text"]
        ocr_rows.append({
            **identity,
            "ocr_text": text,
            "ocr_text_normalized": normalize_search(text),
            "ocr_text_no_accent": remove_accents(normalize_search(text)),
            "has_text": bool(regions),
            "ocr_status": result["ocr_status"],
            "region_count": len(regions),
            "dropped_region_count": result["dropped_region_count"],
            "image_width": result["image_width"],
            "image_height": result["image_height"],
            "spotting_format": result["spotting_format"],
            "finish_reason": result["finish_reason"] or "",
            "completion_tokens": result["completion_tokens"],
            "raw_output": result["raw_output"],
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        })
        for region in regions:
            region_rows.append({
                **identity,
                "region_id": region["region_id"],
                "reading_order": region["reading_order"],
                "text": region["text"],
                "text_normalized": normalize_search(region["text"]),
                "text_no_accent": remove_accents(normalize_search(region["text"])),
                "bbox_x1": region["bbox_x1"],
                "bbox_y1": region["bbox_y1"],
                "bbox_x2": region["bbox_x2"],
                "bbox_y2": region["bbox_y2"],
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            })
    ocr_frame = pd.DataFrame(ocr_rows, columns=OCR_COLUMNS)
    region_frame = pd.DataFrame(region_rows, columns=REGION_COLUMNS)
    for column in [
        "sample_n", "frame_idx", "shot_id", "region_count", "dropped_region_count",
        "image_width", "image_height", "completion_tokens",
    ]:
        ocr_frame[column] = ocr_frame[column].astype("int64")
    ocr_frame["timestamp_sec"] = ocr_frame["timestamp_sec"].astype("float64")
    ocr_frame["has_text"] = ocr_frame["has_text"].astype("bool")
    for column in [
        "video_id", "frame_uid", "image_path", "ocr_text", "ocr_text_normalized",
        "ocr_text_no_accent", "ocr_status", "spotting_format", "finish_reason",
        "raw_output", "model_id", "model_revision",
    ]:
        ocr_frame[column] = ocr_frame[column].astype("string")
    for column in ["sample_n", "frame_idx", "shot_id", "region_id", "reading_order"]:
        region_frame[column] = region_frame[column].astype("int64")
    for column in ["timestamp_sec", "bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]:
        region_frame[column] = region_frame[column].astype("float64")
    for column in [
        "video_id", "frame_uid", "image_path", "text", "text_normalized",
        "text_no_accent", "model_id", "model_revision",
    ]:
        region_frame[column] = region_frame[column].astype("string")
    return ocr_frame, region_frame


def validate_artifacts(manifest, ocr_frame, region_frame):
    if list(ocr_frame["frame_uid"]) != list(manifest["frame_uid"]):
        raise RuntimeError("ocr.parquet identity/order mismatch")
    if not set(ocr_frame["ocr_status"]).issubset(OCR_STATUSES):
        raise RuntimeError("Unexpected OCR status")
    actual_counts = region_frame.groupby("frame_uid").size().to_dict()
    expected_counts = dict(zip(ocr_frame["frame_uid"], ocr_frame["region_count"]))
    if any(actual_counts.get(uid, 0) != count for uid, count in expected_counts.items()):
        raise RuntimeError("region_count does not match ocr_regions.parquet")
    if (ocr_frame["has_text"] != ocr_frame["region_count"].gt(0)).any():
        raise RuntimeError("has_text does not match region_count")
    if not region_frame.empty:
        boxes = region_frame[["bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]]
        if not ((boxes >= 0).all().all() and (boxes <= 1).all().all()):
            raise RuntimeError("Normalized OCR box is outside [0, 1]")
        if not (
            (region_frame["bbox_x2"] > region_frame["bbox_x1"]).all()
            and (region_frame["bbox_y2"] > region_frame["bbox_y1"]).all()
        ):
            raise RuntimeError("OCR region has non-positive area")
        if region_frame["text"].str.strip().eq("").any():
            raise RuntimeError("OCR region has empty text")


def write_artifacts(spec, manifest, results, stats):
    ocr_frame, region_frame = build_artifact_frames(manifest, results)
    validate_artifacts(manifest, ocr_frame, region_frame)
    directory = OUTPUT_ROOT / spec["level"] / spec["video_id"]
    directory.mkdir(parents=True, exist_ok=True)
    parquet_path = directory / "ocr.parquet"
    regions_path = directory / "ocr_regions.parquet"
    marker_path = directory / "_OCR_SUCCESS.json"
    ocr_frame.to_parquet(parquet_path, index=False, compression="zstd")
    region_frame.to_parquet(regions_path, index=False, compression="zstd")
    marker = {
        "status": "complete",
        "video_id": spec["video_id"],
        "source_repo": INPUT_REPO,
        "source_revision": INPUT_REVISION,
        "source_frames_path": spec["frames_filename"],
        "source_keyframes_path": spec["tar_filename"],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "vllm_version": VLLM_VERSION,
        "torch_version": TORCH_VERSION,
        "transformers_version": TRANSFORMERS_VERSION,
        "dtype": VLLM_DTYPE,
        "task_type": TASK_TYPE,
        "run_config": RUN_CONFIG,
        "run_config_sha256": RUN_CONFIG_SHA256,
        "frame_count": len(ocr_frame),
        "frames_with_text": int(ocr_frame["has_text"].sum()),
        "region_count": len(region_frame),
        "status_counts": {
            key: int(value) for key, value in ocr_frame["ocr_status"].value_counts().items()
        },
        "parquet_sha256": sha256_file(parquet_path),
        "regions_parquet_sha256": sha256_file(regions_path),
        "stats": stats,
        "schema_version": SCHEMA_VERSION,
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    marker_path.write_text(json.dumps(marker, ensure_ascii=False, indent=2), encoding="utf-8")
    return [parquet_path, regions_path, marker_path]


def fetch_inputs(spec):
    video_id = spec["video_id"]
    local_dir = DOWNLOAD_ROOT / video_id
    shutil.rmtree(local_dir, ignore_errors=True)
    try:
        frames_path = download_input(spec["frames_filename"], local_dir)
        marker_path = download_input(spec["source_marker"], local_dir)
        source_marker = json.loads(marker_path.read_text(encoding="utf-8"))
        manifest = make_manifest(pd.read_parquet(frames_path), video_id)
        if (
            source_marker.get("video_id") != video_id
            or source_marker.get("keyframe_count") != len(manifest)
        ):
            raise RuntimeError(f"{video_id}: invalid keyframe source marker")
        tar_path = download_input(spec["tar_filename"], local_dir)
    except BaseException:
        shutil.rmtree(local_dir, ignore_errors=True)
        raise
    return {"local_dir": local_dir, "manifest": manifest, "tar_path": tar_path}


def discard_prefetch(future):
    if future is None or future.cancel():
        return
    try:
        shutil.rmtree(future.result()["local_dir"], ignore_errors=True)
    except Exception:
        pass


def collect_qa_samples(archive, manifest, results):
    by_path, by_name = tar_lookup(archive)
    text_positions = [i for i, item in enumerate(results) if item["regions"]]
    other_positions = [i for i, item in enumerate(results) if not item["regions"]]
    positions = list(dict.fromkeys(
        spread_indices(text_positions, 6) + spread_indices(other_positions, 2)
    ))[:8]
    samples = []
    for position in positions:
        row = manifest.iloc[position]
        image_bytes = read_frame_bytes(archive, row["image_path"], by_path, by_name)
        samples.append({
            "frame_uid": row["frame_uid"],
            "timestamp_sec": float(row["timestamp_sec"]),
            "ocr_text": results[position]["ocr_text"],
            "ocr_status": results[position]["ocr_status"],
            "region_count": len(results[position]["regions"]),
            "jpeg": preview_jpeg(image_bytes, results[position]["regions"]),
        })
    return samples


def process_video(spec, inputs):
    video_id = spec["video_id"]
    manifest = inputs["manifest"]
    started = time.perf_counter()
    try:
        with tarfile.open(inputs["tar_path"], mode="r:*") as archive:
            results, inference_sec, resumed_frames = infer_video(archive, manifest, video_id)
            qa_samples = (
                collect_qa_samples(archive, manifest, results)
                if MAX_VIDEOS_PER_RUN is not None else []
            )
        inferred_frames = len(results) - resumed_frames
        status_counts = Counter(item["ocr_status"] for item in results)
        stats = {
            "elapsed_sec": round(time.perf_counter() - started, 3),
            "inference_sec": round(inference_sec, 3),
            "inference_frames": inferred_frames,
            "resumed_frames": resumed_frames,
            "inference_fps": round(inferred_frames / inference_sec, 3) if inference_sec else None,
            "completion_tokens": sum(item["completion_tokens"] for item in results),
            "servers": len(SERVERS),
            "gpus": [gpu["name"] for gpu in GPUS if gpu["index"] in GPU_IDS],
        }
        paths = write_artifacts(spec, manifest, results, stats)
        return {
            "video_id": video_id,
            "paths": paths,
            "frames": len(manifest),
            "regions": sum(len(item["regions"]) for item in results),
            "frames_with_text": sum(bool(item["regions"]) for item in results),
            "status_counts": dict(status_counts),
            "stats": stats,
            "qa_samples": qa_samples,
        }
    finally:
        shutil.rmtree(inputs["local_dir"], ignore_errors=True)
        gc.collect()


def upload_outputs(paths, video_id):
    operations = [
        CommitOperationAdd(
            path_in_repo=f"data/{path.relative_to(OUTPUT_ROOT).as_posix()}",
            path_or_fileobj=str(path),
        )
        for path in paths
    ]
    retry(
        lambda: api.create_commit(
            repo_id=OUTPUT_REPO,
            repo_type="dataset",
            operations=operations,
            commit_message=f"Add HunyuanOCR-1.5 OCR for {video_id}",
        ),
        f"upload {video_id}",
    )


def throughput_probe(archive, manifest):
    by_path, by_name = tar_lookup(archive)
    positions = spread_indices(list(range(len(manifest))), THROUGHPUT_PROBE_FRAMES)
    payloads = [
        read_frame_bytes(archive, manifest.iloc[p]["image_path"], by_path, by_name)
        for p in positions
    ]
    started = time.perf_counter()
    futures = [
        REQUEST_POOL.submit(ocr_frame, payload, SERVERS[index % len(SERVERS)])
        for index, payload in enumerate(payloads)
    ]
    results = [future.result() for future in futures]
    elapsed = time.perf_counter() - started
    fps = len(results) / elapsed
    print(
        f"\nThroughput probe: {len(results)} frames in {elapsed:.1f}s → {fps:.2f} frames/s "
        f"on {len(SERVERS)} GPU(s)"
    )
    print(f"  ≈ {100_000 / fps / 3600:.1f} giờ cho mỗi 100k keyframe với số GPU này")
    print("  Trạng thái:", dict(Counter(item["ocr_status"] for item in results)))
    print("  Token output trung bình:", round(sum(i["completion_tokens"] for i in results) / len(results), 1))


def run_preflight():
    if PREFLIGHT_FRAMES < len(PREFLIGHT_FRAME_SPECS):
        raise ValueError("PREFLIGHT_FRAMES is smaller than required frame specs")
    grouped = defaultdict(list)
    for video_id, frame_idx in PREFLIGHT_FRAME_SPECS:
        grouped[video_id].append(frame_idx)
    selected, opened, samples = [], {}, []
    try:
        for video_id, frame_indices in grouped.items():
            spec = spec_by_id.get(video_id)
            if spec is None:
                raise RuntimeError(f"Preflight video not found: {video_id}")
            local_dir = DOWNLOAD_ROOT / f"preflight-{video_id}"
            shutil.rmtree(local_dir, ignore_errors=True)
            frames_path = download_input(spec["frames_filename"], local_dir)
            tar_path = download_input(spec["tar_filename"], local_dir)
            manifest = make_manifest(pd.read_parquet(frames_path), video_id)
            positions = []
            for frame_idx in frame_indices:
                matches = manifest.index[manifest["frame_idx"] == frame_idx].tolist()
                if not matches:
                    raise RuntimeError(f"Preflight frame not found: {video_id}:{frame_idx}")
                positions.append(matches[0])
            opened[video_id] = (local_dir, tar_path, manifest, positions)
            selected.extend((video_id, position) for position in positions)
        first_video = next(iter(opened))
        _, first_tar, first_manifest, used = opened[first_video]
        remaining = [i for i in range(len(first_manifest)) if i not in used]
        selected.extend(
            (first_video, i)
            for i in spread_indices(remaining, PREFLIGHT_FRAMES - len(selected))
        )

        for video_id, position in selected:
            _, tar_path, manifest, _ = opened[video_id]
            with tarfile.open(tar_path, mode="r:*") as archive:
                by_path, by_name = tar_lookup(archive)
                row = manifest.iloc[position]
                image_bytes = read_frame_bytes(archive, row["image_path"], by_path, by_name)
            result = ocr_frame(image_bytes, SERVERS[0])
            samples.append({
                "frame_uid": row["frame_uid"],
                "timestamp_sec": float(row["timestamp_sec"]),
                "result": result,
                "jpeg": preview_jpeg(image_bytes, result["regions"]),
            })
        with tarfile.open(first_tar, mode="r:*") as archive:
            throughput_probe(archive, first_manifest)
    finally:
        for local_dir, _, _, _ in opened.values():
            shutil.rmtree(local_dir, ignore_errors=True)

    from IPython.display import Image as DisplayImage
    from IPython.display import display
    print(f"\nPreflight: {len(samples)} frames; nothing will be uploaded")
    for sample in samples:
        result = sample["result"]
        print(
            "\n", sample["frame_uid"], f"t={sample['timestamp_sec']:.2f}s",
            f"latency={result['latency_sec']:.2f}s", "status=", result["ocr_status"],
            "format=", result["spotting_format"], "regions=", len(result["regions"]),
            "tokens=", result["completion_tokens"],
        )
        print("OCR:", result["ocr_text"] or NO_TEXT_SENTINEL)
        if result["ocr_status"] in {"parse_error", "repetition", "truncated"}:
            print("RAW:", result["raw_output"][:1500])
        display(DisplayImage(data=sample["jpeg"]))
    statuses = [sample["result"]["ocr_status"] for sample in samples]
    if all(status != "ok" for status in statuses):
        print(
            "\nCẢNH BÁO: không frame nào đọc được chữ. Nếu ảnh rõ ràng có chữ, kiểm tra "
            "RAW output; với float16 trên T4 hãy thử VLLM_DTYPE='float32'."
        )
    return [{key: value for key, value in sample.items() if key != "jpeg"} for sample in samples]


preflight_results = run_preflight() if PREFLIGHT_ONLY or RUN_SMOKE_TEST_BEFORE_INGEST else []
if not PREFLIGHT_ONLY and RUN_SMOKE_TEST_BEFORE_INGEST:
    print("Smoke test passed; starting queued video ingestion.")
if PREFLIGHT_ONLY:
    pending_specs = []
    print("Preflight finished. Review text, boxes and throughput, then set PREFLIGHT_ONLY=False.")

run_results, failures = [], []
uploaded_video_ids = set()


def finalize_upload(result):
    video_id = result["video_id"]
    upload_outputs(result["paths"], video_id)
    shutil.rmtree(OUTPUT_ROOT / video_id.split("_")[0] / video_id, ignore_errors=True)
    checkpoint_path(video_id).unlink(missing_ok=True)
    return video_id


def collect_upload(upload_futures, future):
    result = upload_futures.pop(future)
    try:
        video_id = future.result()
    except Exception as error:
        failures.append({"video_id": result["video_id"], "stage": "upload", "error": repr(error)})
        print(f"UPLOAD FAILED {result['video_id']}: {error!r}")
    else:
        uploaded_video_ids.add(video_id)
        print(f"Uploaded {video_id}; local artifact and checkpoint removed")


run_started = time.perf_counter()
with (
    ThreadPoolExecutor(max_workers=1) as prefetch_executor,
    ThreadPoolExecutor(max_workers=UPLOAD_WORKERS) as upload_executor,
):
    upload_futures = {}
    last_failure_signature = None
    identical_failures = 0
    next_fetch = prefetch_executor.submit(fetch_inputs, pending_specs[0]) if pending_specs else None
    for index, spec in enumerate(pending_specs):
        current_fetch = next_fetch
        next_fetch = (
            prefetch_executor.submit(fetch_inputs, pending_specs[index + 1])
            if index + 1 < len(pending_specs) else None
        )
        stage = "download"
        try:
            inputs = current_fetch.result()
            stage = "inference"
            result = process_video(spec, inputs)
        except Exception as error:
            signature = f"{type(error).__name__}:{error}"
            identical_failures = identical_failures + 1 if signature == last_failure_signature else 1
            last_failure_signature = signature
            failures.append({"video_id": spec["video_id"], "stage": stage, "error": repr(error)})
            print(f"[{index + 1}/{len(pending_specs)}] FAILED {spec['video_id']} ({stage}): {error!r}")
            if isinstance(error, FatalRequestError) or identical_failures >= MAX_CONSECUTIVE_IDENTICAL_FAILURES:
                print("Circuit breaker: stopping new work; fix the error above, then rerun.")
                discard_prefetch(next_fetch)
                next_fetch = None
                break
            continue
        last_failure_signature = None
        identical_failures = 0
        run_results.append(result)
        stats = result["stats"]
        print(
            f"[{index + 1}/{len(pending_specs)}] OK {result['video_id']}: "
            f"{result['frames']} frames, {result['regions']} regions, "
            f"{stats['inference_fps']} FPS, status {result['status_counts']}"
        )
        upload_futures[upload_executor.submit(finalize_upload, result)] = result
        while len(upload_futures) >= MAX_PENDING_UPLOADS:
            collect_upload(upload_futures, next(as_completed(upload_futures)))
    discard_prefetch(next_fetch)
    while upload_futures:
        collect_upload(upload_futures, next(as_completed(upload_futures)))

print("vLLM servers:", len(SERVERS))
print("Uploaded this run:", len(uploaded_video_ids))
print("Processed this run:", len(run_results), "| Failures:", len(failures))
if run_results:
    total = sum(item["frames"] for item in run_results)
    print("OCR frames:", total, "| overall FPS:", round(total / max(time.perf_counter() - run_started, 1e-6), 3))
if failures:
    print(json.dumps(failures, ensure_ascii=False, indent=2))
    raise RuntimeError("OCR run has download/inference/upload failures; rerun to resume")

## 6. Audit completion và xem mẫu

Full run chỉ hoàn tất khi mọi thành viên nguồn thực tế có marker hợp lệ và hash
khớp cả hai artifact. Pilot hiển thị frame có/không có chữ kèm box để kiểm tra dấu
tiếng Việt, chữ bịa trên frame không có chữ và tỉ lệ `truncated`/`repetition`
trước khi bỏ giới hạn video.

In [ ]:
final_files = set() if PREFLIGHT_ONLY else set(retry(
    lambda: api.list_repo_files(OUTPUT_REPO, repo_type="dataset"),
    "refresh OCR files",
))
with ThreadPoolExecutor(max_workers=12) as validator:
    final_flags = list(validator.map(lambda spec: remote_complete(spec, final_files), video_specs))
missing = [spec["video_id"] for spec, okay in zip(video_specs, final_flags) if not okay]
if not PREFLIGHT_ONLY:
    print(f"Remote complete: {len(video_specs) - len(missing)}/{len(video_specs)} videos")
    print("Dataset:", f"https://huggingface.co/datasets/{OUTPUT_REPO}")
    if missing:
        print("Remaining:", len(missing), "sample:", missing[:20])
selected_ids = {spec["video_id"] for spec in selected_specs}
selected_missing = [video_id for video_id in missing if video_id in selected_ids]
if selected_missing and not PREFLIGHT_ONLY:
    raise RuntimeError(f"Selected OCR videos lack valid output: {selected_missing}")
if MAX_VIDEOS_PER_RUN is None and missing and not PREFLIGHT_ONLY:
    raise RuntimeError(f"Missing valid OCR output for {len(missing)} supplied videos")
if PREFLIGHT_ONLY:
    print("Preflight-only run complete; no artifact was uploaded.")
elif MAX_VIDEOS_PER_RUN is not None:
    print("Pilot complete. Review the boxed images below, then set MAX_VIDEOS_PER_RUN=None.")
    from IPython.display import Image as DisplayImage
    from IPython.display import display
    for result in run_results:
        print(
            "\nQA:", result["video_id"], "| frames:", result["frames"],
            "| with text:", result["frames_with_text"], "| regions:", result["regions"],
            "| status:", result["status_counts"],
        )
        for sample in result["qa_samples"]:
            print(
                sample["frame_uid"], f"t={sample['timestamp_sec']:.2f}s",
                "status:", sample["ocr_status"], "regions:", sample["region_count"],
                "OCR:", sample["ocr_text"] or NO_TEXT_SENTINEL,
            )
            display(DisplayImage(data=sample["jpeg"]))